In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.use_database('F1_ANALYTICS')
session.use_schema('GOLD')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from snowflake.snowpark.functions import col

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# -------------------------------
# LOAD DATA
# -------------------------------
df = session.table('F1_ANALYTICS.GOLD.QUALIFICATION_RESULTS').where(col('session_key') == 11249).to_pandas()

# -------------------------------
# SELECT FEATURES (SIMPLIFIED)
# -------------------------------
feature_cols = ['AVG_QUALI_TIME', 'FASTEST_QUALIFYING_TIME']
df = df.dropna(subset=feature_cols)

X = df[feature_cols]

# -------------------------------
# SCALE FEATURES
# -------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -------------------------------
# K-MEANS CLUSTERING
# -------------------------------
kmeans = KMeans(n_clusters=5, random_state=42)
df['cluster'] = kmeans.fit_predict(X_scaled)

# -------------------------------
# PERFORMANCE SCORE
# -------------------------------
df['performance_score'] = 1 / df['AVG_QUALI_TIME']

# -------------------------------
# MAP CLUSTERS → WIN PROBABILITY
# -------------------------------
cluster_perf = df.groupby('cluster')['performance_score'].mean()
sorted_clusters = cluster_perf.sort_values(ascending=False)

cluster_mapping = {
    sorted_clusters.index[0]: 'Best Chance',
    sorted_clusters.index[1]: 'High Chance',
    sorted_clusters.index[2]: 'Moderate Chance',
    sorted_clusters.index[3]: 'Low Chance',
    sorted_clusters.index[4]: 'Very Low Chance'
    
}

df['win_category'] = df['cluster'].map(cluster_mapping)

# -------------------------------
# OPTIONAL VALIDATION
# -------------------------------
if 'POSITION' in df.columns:
    df['winner_flag'] = (df['POSITION'] == 1).astype(int)
    print(df.groupby('win_category')['winner_flag'].mean())

# -------------------------------
# VISUALIZATION (DIRECT FEATURES)
# -------------------------------
plt.figure(figsize=(10, 6))

for label in df['win_category'].unique():
    subset = df[df['win_category'] == label]
    plt.scatter(
        subset['AVG_QUALI_TIME'],
        subset['FASTEST_QUALIFYING_TIME'],
        label=label
    )
    # Add driver_number labels
    for _, row in subset.iterrows():
        plt.text(
            row['AVG_QUALI_TIME'],
            row['FASTEST_QUALIFYING_TIME'],
            str(row['DRIVER_NUMBER']),
            fontsize=9,
            ha='right',   # horizontal alignment
            va='bottom'   # vertical alignment
        )

plt.xlabel("Average Qualifying Time")
plt.ylabel("Fastest Qualifying Time")
plt.title("Driver Clusters (Simple Model)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import silhouette_score
n_clusters =  5
# -------------------------------
# SILHOUETTE SCORE
# -------------------------------
if n_clusters > 1:  # silhouette score requires at least 2 clusters
    score = silhouette_score(X_scaled, df['cluster'])
    print(f"Silhouette Score: {score:.3f}")
else:
    print("Silhouette score not applicable for a single cluster.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from adjustText import adjust_text
fig, ax = plt.subplots(figsize=(13, 8))
# Style
fig.patch.set_facecolor('#0f0f1a')
ax.set_facecolor('#0f0f1a')
ax.grid(True, color='#ffffff15', linewidth=0.8, zorder=0)
for spine in ax.spines.values():
   spine.set_edgecolor('#ffffff20')
ax.tick_params(colors='#aaaacc', labelsize=9)
# F1-inspired color palette
palette = ['#e8002d', '#00d2be', '#ff8700', '#ffffff', '#0067ff',
          '#900000', '#006f62', '#2b4562', '#b6babd', '#ff87bc']
texts = []
for i, label in enumerate(df['win_category'].unique()):
   subset = df[df['win_category'] == label]
   color = palette[i % len(palette)]
   ax.scatter(
       subset['AVG_QUALI_TIME'],
       subset['FASTEST_QUALIFYING_TIME'],
       label=label,
       color=color,
       s=90,
       edgecolors='white',
       linewidths=0.5,
       zorder=3,
       alpha=0.92
   )
   if 'FULL_NAME' in df.columns:
       for _, row in subset.iterrows():
           t = ax.text(
               row['AVG_QUALI_TIME'],
               row['FASTEST_QUALIFYING_TIME'],
               str(row['FULL_NAME']),
               fontsize=8.5,
               color=color,
               fontweight='semibold',
               zorder=5
           )
           # White outline for readability on dark bg
           t.set_path_effects([
               pe.withStroke(linewidth=2.5, foreground='#0f0f1a')
           ])
           texts.append(t)
# Auto-adjust labels to prevent overlaps
adjust_text(
   texts,
   ax=ax,
   expand_points=(1.6, 1.6),
   expand_text=(1.4, 1.4),
   arrowprops=dict(arrowstyle='-', color='#ffffff40', lw=0.8),
   force_text=(0.5, 0.8),
   force_points=(0.3, 0.5),
   avoid_self=True,
)
ax.set_xlabel("Average Qualifying Time", color='#aaaacc', fontsize=11, labelpad=10)
ax.set_ylabel("Fastest Qualifying Time", color='#aaaacc', fontsize=11, labelpad=10)
ax.set_title("Driver Clusters", color='white', fontsize=15, fontweight='bold', pad=16)
legend = ax.legend(
   framealpha=0.2,
   facecolor='#1a1a2e',
   edgecolor='#ffffff30',
   labelcolor='white',
   fontsize=9,
   title='Win Category',
   title_fontsize=9
)
legend.get_title().set_color('#aaaacc')
plt.tight_layout()
plt.show()